# Treinamento YOLO Pose com K-Fold (Colab)

Este notebook assume que o dataset já está preparado em formato YOLO (images/labels) dentro de `dataset/fold_*`.

In [8]:
!pip install -q ultralytics==8.4.0

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configure os caminhos

Ajuste os caminhos abaixo para o seu Drive/ambiente Colab.

In [14]:
from pathlib import Path

DATASET_ROOT = Path('/content/drive/MyDrive/cow-classifier/dataset')
MODELS_DIR = Path('/content/drive/MyDrive/cow-classifier/models')
# BASE_MODEL = 'yolo26n-pose.pt'
# BASE_MODEL = 'yolo26l-pose.pt'
BASE_MODEL = 'yolo26x-pose.pt'
PROJECT_DIR = Path('/content/drive/MyDrive/cow-classifier/runs/kfold_pose')
EPOCHS = 100
IMGSZ = 640 # Reduzido para economizar memória da GPU
BATCH = 8 # alterado de 8 para 32
DEVICE = 0  # 0 para GPU no Colab, ou 'cpu'
WORKERS = 16 # incrementado a quantidade de works de 2 para 16
RUN_PREFIX = 'train'
CONTINUE_FROM_BEST = False
PATIENCE = 50
SAVE_PERIOD = 10
CONF_MIN = 0.30

In [11]:
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
# print(f'Diretório de modelos: {MODELS_DIR}')
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt -P {MODELS_DIR}
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26l.pt -P {MODELS_DIR}
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26x.pt -P {MODELS_DIR}
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n-pose.pt -P {MODELS_DIR}
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26l-pose.pt -P {MODELS_DIR}
# !wget -nc https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26x-pose.pt -P {MODELS_DIR}

In [12]:
import csv
import json
from datetime import datetime
from statistics import mean
from ultralytics import YOLO
import time # Importar o módulo time

def find_fold_yaml_files(dataset_root: Path):
    fold_yaml_files = []
    for fold_dir in sorted(dataset_root.glob('fold_*')):
        if not fold_dir.is_dir():
            continue
        yaml_candidates = sorted(fold_dir.glob('data_fold_*.yaml')) + sorted(fold_dir.glob('data_fold_*.yml'))
        if yaml_candidates:
            fold_yaml_files.append((fold_dir.name, yaml_candidates[0]))
    return fold_yaml_files

def read_metric(train_result, metric_key):
    results_dict = getattr(train_result, 'results_dict', None)
    if isinstance(results_dict, dict):
        return results_dict.get(metric_key)
    return None

def read_first_metric(train_result, keys):
    for key in keys:
        value = read_metric(train_result, key)
        if value is not None:
            return value
    return None

def existing_best_checkpoint(project_dir: Path, run_name: str):
    best_path = project_dir / run_name / 'weights' / 'best.pt'
    return best_path if best_path.exists() else None

def read_best_epoch_stats(results_csv_path: Path):
    if not results_csv_path.exists():
        return {}
    with results_csv_path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    pose_key = 'metrics/mAP50-95(P)'
    box_key = 'metrics/mAP50-95(B)'
    selected_key = pose_key if pose_key in rows[0] else box_key
    best_row = None
    best_value = float('-inf')
    for row in rows:
        raw = row.get(selected_key)
        if raw in (None, ''):
            continue
        value = float(raw)
        if value > best_value:
            best_value = value
            best_row = row
    if best_row is None:
        return {}
    return {
        'best_epoch': int(float(best_row.get('epoch', 0))),
        'metric_used': selected_key,
        'metric_value': best_value,
    }

def mean_or_none(values):
    valid = [v for v in values if isinstance(v, (int, float))]
    return mean(valid) if valid else None

def build_report(summary, conf_min, total_duration_seconds):
    box_map50_values = [item.get('box_map50') for item in summary]
    box_map5095_values = [item.get('box_map50_95') for item in summary]
    pose_map50_values = [item.get('pose_map50') for item in summary]
    pose_map5095_values = [item.get('pose_map50_95') for item in summary]

    pose_best_fold = None
    best_pose_value = float('-inf')
    for item in summary:
        value = item.get('pose_map50_95')
        if isinstance(value, (int, float)) and value > best_pose_value:
            best_pose_value = value
            pose_best_fold = item

    return {
        'k_folds': len(summary),
        'total_duration_seconds': total_duration_seconds, # Adicionar tempo total de execução
        'aggregate_metrics': {
            'Box_mAP50_media_folds': mean_or_none(box_map50_values),
            'Box_mAP50_95_media_folds': mean_or_none(box_map5095_values),
            'Pose_mAP50_media_folds': mean_or_none(pose_map50_values),
            'Pose_mAP50_95_media_folds': mean_or_none(pose_map5095_values),
            'Pose_mAP50_95_melhor_fold': {
                'valor': None if pose_best_fold is None else pose_best_fold.get('pose_map50_95'),
                'fold': None if pose_best_fold is None else pose_best_fold.get('fold'),
            },
        },
        'metricas_teste_final': {
            'accuracy': None,
            'f1_macro': None,
            'top1_accuracy': None,
            'top3_accuracy': None,
            'top5_accuracy': None,
        },
        'metricas_com_rejeicao': {
            'confianca_min': conf_min,
            'cobertura': None,
            'accuracy_aceitas': None,
            'f1_macro_aceitas': None,
        },
        'folds': summary,
    }

def train_kfold(dataset_root, models_dir, base_model, project_dir, epochs, imgsz, batch, device, workers, run_prefix, continue_from_best, patience, save_period, conf_min):
    dataset_root = Path(dataset_root)
    models_dir = Path(models_dir)
    project_dir = Path(project_dir)
    base_model_path = models_dir / base_model

    if not dataset_root.exists():
        raise FileNotFoundError(f'Dataset root não encontrado: {dataset_root}')
    if not base_model_path.exists():
        raise FileNotFoundError(f'Modelo base não encontrado: {base_model_path}')

    fold_yaml_files = find_fold_yaml_files(dataset_root)
    if not fold_yaml_files:
        raise FileNotFoundError(f'Nenhum data_fold_*.yaml encontrado em {dataset_root}/fold_*')

    project_dir.mkdir(parents=True, exist_ok=True)
    print(f'Total de folds: {len(fold_yaml_files)}')

    summary = []
    start_time_total = time.time() # Iniciar contagem de tempo total

    for fold_name, data_yaml in fold_yaml_files:
        run_name = f'{run_prefix}_{fold_name}'
        model_start_path = base_model_path
        best_checkpoint = existing_best_checkpoint(project_dir, run_name)
        if continue_from_best and best_checkpoint is not None:
            model_start_path = best_checkpoint

        print(f'\n--- Iniciando {fold_name} ---')
        model = YOLO(str(model_start_path))
        result = model.train(
            data=str(data_yaml),
            task='pose',
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            workers=workers,
            device=device,
            project=str(project_dir),
            name=run_name,
            save=True,
            save_period=save_period,
            patience=patience,
            exist_ok=True,
        )

        save_dir = Path(getattr(result, 'save_dir', project_dir / run_name))
        summary.append({
            'fold': fold_name,
            'run': run_name,
            'box_map50': read_first_metric(result, ['metrics/mAP50(B)']),
            'box_map50_95': read_first_metric(result, ['metrics/mAP50-95(B)']),
            'pose_map50': read_first_metric(result, ['metrics/mAP50(P)']),
            'pose_map50_95': read_first_metric(result, ['metrics/mAP50-95(P)']),
            'best_weights': str(save_dir / 'weights' / 'best.pt'),
            'last_weights': str(save_dir / 'weights' / 'last.pt'),
            'best_epoch_stats': read_best_epoch_stats(save_dir / 'results.csv'),
        })

    end_time_total = time.time() # Finalizar contagem de tempo total
    total_duration_seconds = end_time_total - start_time_total

    report = build_report(summary, conf_min, total_duration_seconds)
    report_path = project_dir / 'kfold_metrics_report.json'
    payload = {
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'settings': {
            'dataset_root': str(dataset_root),
            'base_model': str(base_model_path),
            'continue_from_best': continue_from_best,
            'epochs': epochs,
            'imgsz': imgsz,
            'batch': batch,
            'device': device,
            'workers': workers,
            'project': str(project_dir),
            'run_prefix': run_prefix,
            'save_period': save_period,
            'patience': patience,
        },
        'report': report,
    }
    with report_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f'\nRelatório salvo em: {report_path}')
    return report_path

In [15]:
report_path = train_kfold(
    dataset_root=DATASET_ROOT,
    models_dir=MODELS_DIR,
    base_model=BASE_MODEL,
    project_dir=PROJECT_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    run_prefix=RUN_PREFIX,
    continue_from_best=CONTINUE_FROM_BEST,
    patience=PATIENCE,
    save_period=SAVE_PERIOD,
    conf_min=CONF_MIN,
)
report_path

Total de folds: 5

--- Iniciando fold_0 ---
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cow-classifier/dataset/fold_0/data_fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/d

/tmp/ipykernel_10724/3417919935.py:175: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': datetime.utcnow().isoformat() + 'Z',


PosixPath('/content/drive/MyDrive/cow-classifier/runs/kfold_pose/kfold_metrics_report.json')

## Testando o Modelo Treinado e Fazendo Predições

Agora que o treinamento K-Fold foi concluído e o relatório gerado, podemos testar um dos modelos treinados e usá-lo para fazer predições em novas imagens.

In [16]:
import json
from ultralytics import YOLO
from pathlib import Path

# Carregar o relatório K-Fold para obter o caminho do melhor modelo
# Certifique-se de que report_path esteja definido ou use o caminho direto
if 'report_path' not in locals():
    # Se report_path não foi definido, tentamos encontrá-lo com base em PROJECT_DIR
    report_path = PROJECT_DIR / 'kfold_metrics_report.json'

if report_path.exists():
    with open(report_path, 'r', encoding='utf-8') as f:
        kfold_report = json.load(f)

    # Pegar o caminho do melhor modelo do primeiro fold como exemplo
    # Você pode adaptar esta lógica para selecionar o melhor modelo de todos os folds
    best_model_path = None
    if kfold_report and kfold_report['report']['folds']:
        # Obter o melhor modelo do fold com a maior métrica pose_mAP50_95
        best_fold_summary = kfold_report['report']['aggregate_metrics']['Pose_mAP50_95_melhor_fold']['fold']
        for fold_data in kfold_report['report']['folds']:
            if fold_data['fold'] == best_fold_summary:
                best_model_path = fold_data['best_weights']
                print(f"Caminho do melhor modelo (Fold {best_fold_summary}): {best_model_path}")
                break

    if best_model_path:
        # Carregar o modelo treinado
        model_for_test = YOLO(best_model_path)
        print("Modelo treinado carregado com sucesso para teste e predição.")
    else:
        print("Nenhum modelo treinado encontrado no relatório. Verifique o caminho do relatório ou a estrutura.")
else:
    print(f"Relatório K-Fold não encontrado em: {report_path}")
    print("Por favor, execute o treinamento K-Fold primeiro para gerar o relatório.")


Caminho do melhor modelo (Fold fold_1): /content/drive/MyDrive/cow-classifier/runs/kfold_pose/train_fold_1/weights/best.pt
Modelo treinado carregado com sucesso para teste e predição.


In [17]:
if 'model_for_test' in locals() and model_for_test is not None:
    print("\n--- Executando validação com o melhor modelo treinado ---")
    # Você pode precisar especificar o 'data' YAML para a validação,
    # idealmente, o conjunto de teste de um dos folds ou um conjunto de teste separado.
    # Por simplicidade, vamos usar o YAML do fold que gerou o melhor modelo.
    best_fold_data_yaml = Path(kfold_report['settings']['dataset_root']) / best_fold_summary / f"data_{best_fold_summary}.yaml"

    metrics = model_for_test.val(
        data=str(best_fold_data_yaml),
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        device=DEVICE,
        conf=CONF_MIN, # Usar o CONF_MIN definido anteriormente
        project=str(PROJECT_DIR), # Salvar resultados da validação no mesmo diretório do projeto
        name=f'val_{best_fold_summary}', # Nome para a execução de validação
        save_json=True # Salvar métricas em formato JSON
    )
    print("Validação concluída.")
    print(f"mAP50-95(P) na validação: {metrics.results_dict.get('metrics/mAP50-95(P)')}")
else:
    print("Modelo não carregado. Não é possível executar a validação.")



--- Executando validação com o melhor modelo treinado ---
Ultralytics 8.4.0 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26x-pose summary (fused): 200 layers, 57,549,975 parameters, 0 gradients, 201.7 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 365.3±154.4 MB/s, size: 730.9 KB)
val: Scanning /content/drive/MyDrive/cow-classifier/dataset/fold_1/labels/val.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 58.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s
                   all        180        180          1          1      0.995      0.953          1          1      0.995      0.939
Speed: 0.6ms preprocess, 37.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Saving /content/drive/MyDrive/cow-classifier/runs/kfold_pose/val_fold_1/predictions.json...
Results saved to /content/drive/MyDrive

In [18]:
if 'model_for_test' in locals() and model_for_test is not None:
    print("\n--- Fazendo predição nas imagens de teste ---")

    # Define o diretório de imagens de teste baseado no melhor fold identificado
    # O caminho segue o padrão: dataset/fold_X/images/test
    test_images_dir = Path(kfold_report['settings']['dataset_root']) / best_fold_summary / 'images' / 'test'

    if test_images_dir.exists() and any(test_images_dir.iterdir()):
        print(f"Executando predições para imagens em: {test_images_dir}")
        results = model_for_test.predict(
            source=str(test_images_dir),
            imgsz=IMGSZ,
            conf=CONF_MIN, # Confiança mínima para detecção
            save=True, # Salvar imagem com predições
            project=str(PROJECT_DIR), # Salvar resultados no diretório do projeto
            name=f'predict_{best_fold_summary}', # Nome para a execução de predição
            show=False, # Não mostrar a imagem aqui, apenas salvar
            save_txt=False, # Não salvar os resultados em um arquivo .txt
            save_conf=True # Salvar a confiança das predições
        )

        print(f"Processamento concluído. {len(results)} imagens processadas.")
        if len(results) > 0 and results[0].save_dir:
             print(f"Resultados salvos em: {results[0].save_dir}")

    else:
        print(f"O diretório de imagens de teste não foi encontrado ou está vazio: {test_images_dir}")
        print("Verifique se as imagens de teste estão na estrutura correta (dataset/fold_*/images/test).")
else:
    print("Modelo não carregado. Não é possível fazer predições.")


--- Fazendo predição nas imagens de teste ---
Executando predições para imagens em: /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test

image 1/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260101_055311_baia19_IPC2.jpg: 320x640 1 cow, 64.7ms
image 2/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260101_211708_baia17_IPC1.jpg: 288x640 1 cow, 65.9ms
image 3/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260102_053609_baia18_IPC1.jpg: 288x640 1 cow, 53.7ms
image 4/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260102_063007_baia23_VIPWX.jpg: 256x640 1 cow, 71.7ms
image 5/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260102_123914_baia14_IPC3.jpg: 288x640 1 cow, 54.6ms
image 6/72 /content/drive/MyDrive/cow-classifier/dataset/fold_1/images/test/20260102_140206_baia23_VIPWX.jpg: 256x640 1 cow, 45.9ms
image 7/72 /content/drive/MyDrive/cow-classifier/dataset/fold

In [19]:
try:
    from google.colab import runtime
    print("Desconectando o ambiente do Colab...")
    runtime.unassign()
except ImportError:
    print("Este código não está rodando no Google Colab.")

Desconectando o ambiente do Colab...


# Task
Configure and run k-fold cross-validation training for a YOLO pose model in Colab, including clearing the `PROJECT_DIR` for new runs and understanding how to use saved models for transfer learning.

## Adicionar Célula de Limpeza

### Subtask:
Criar uma nova célula de código para limpar o diretório `PROJECT_DIR`. Esta etapa removerá os resultados de treinamentos anteriores, garantindo um ambiente limpo para um novo treinamento. Será incluído um aviso antes da exclusão.


**Reasoning**:
The user explicitly asked to add a code cell to clean the `PROJECT_DIR` by removing its contents if it exists. The provided Python code snippet handles this logic, including a warning before deletion.



In [ ]:
# import shutil
# import os

# if PROJECT_DIR.exists():
#    print(f'ATENÇÃO: Removendo o diretório existente: {PROJECT_DIR}')
#    shutil.rmtree(PROJECT_DIR)
#    print('Diretório removido com sucesso.')
# else:
#    print(f'Diretório {PROJECT_DIR} não existe, não é necessário remover.')

**Reasoning**:
The previous attempt to clear the `PROJECT_DIR` failed because the `PROJECT_DIR` variable was not defined in the cell's scope at execution time. To fix this, I will re-include the definition of `PROJECT_DIR` within the cleaning cell itself, ensuring it is always available before attempting to delete the directory.



## Final Task

### Subtask:
Confirmar que a célula de limpeza foi adicionada e que o processo de salvamento de modelos para transfer learning está claro.
